# comfy_colab — one-click SDXL image generation on Colab (free T4 GPU)

A single, self-contained notebook: install ComfyUI, boot it, generate, display.
No external scripts — everything lives in these four cells.

**Runtime → Run all** (or run cells top-to-bottom). First run downloads ~6.5 GB (SDXL), takes ~2 min.
Subsequent runs in the same session are instant.

Edit the **prompt** field in cell 3 and re-run just that cell + cell 4 to iterate.


In [ ]:
#@title 1 · Setup: install ComfyUI + download SDXL { display-mode: "form" }
import os, subprocess

COMFY = "/content/ComfyUI"
CKPT_DIR = f"{COMFY}/models/checkpoints"
CKPT = f"{CKPT_DIR}/sd_xl_base_1.0.safetensors"
SDXL_URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
            "resolve/main/sd_xl_base_1.0.safetensors")

if not os.path.isdir(f"{COMFY}/.git"):
    print("Cloning ComfyUI...")
    subprocess.run(["git", "clone", "-q", "https://github.com/comfyanonymous/ComfyUI.git", COMFY], check=True)
else:
    print("ComfyUI already present")
print("Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"], check=True)

if not os.path.exists(CKPT):
    print("Downloading SDXL base (~6.5 GB)...")
    os.makedirs(CKPT_DIR, exist_ok=True)
    subprocess.run(["wget", "-q", "--show-progress", "-O", CKPT, SDXL_URL], check=True)

print(f"\nSetup done. Checkpoint: {os.path.getsize(CKPT)//1048576} MB")


In [ ]:
#@title 2 · Start ComfyUI server { display-mode: "form" }
import subprocess, time, os, urllib.request

PORT = 8188
PIDFILE = "/content/comfy.pid"
LOGFILE = "/content/comfy.log"

def _alive(pid):
    try:
        os.kill(pid, 0); return True
    except Exception:
        return False

pid = None
if os.path.exists(PIDFILE):
    try: pid = int(open(PIDFILE).read().strip())
    except: pid = None

if pid and _alive(pid):
    print(f"server already running, pid={pid}")
else:
    print("launching server...")
    log = open(LOGFILE, "w")
    subprocess.Popen(
        ["python", "main.py", "--listen", "0.0.0.0", "--port", str(PORT),
         "--output-directory", os.path.join(COMFY, "output")],
        cwd=COMFY, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=True)
    for i in range(90):
        time.sleep(1)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
            print(f"READY on http://127.0.0.1:{PORT}  (boot {i+1}s)")
            break
        except Exception:
            pass
    else:
        print("TIMEOUT - tail of log:")
        print(open(LOGFILE).read()[-1000:])


In [ ]:
#@title 3 · Generate image { display-mode: "form" }
prompt = "A tilted brass arrow mounted on a wooden base casts a crisp shadow onto a horizontal brass rail beneath it, illustrating geometric projection. Cobblestone workshop floor, copper pipes in background, soft window light from the left. Dust motes in the air. Photorealistic steampunk still life, warm tones, detailed metal textures." #@param {type:"string"}
negative = "blurry, low quality, deformed, watermark, text" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
width = 1024 #@param {type:"integer"}
height = 1024 #@param {type:"integer"}
steps = 25 #@param {type:"integer"}
cfg = 7.5 #@param {type:"number"}

import json, time, urllib.request, urllib.parse, os

SERVER = f"http://127.0.0.1:{PORT}"
OUT = f"{COMFY}/output"
os.makedirs(OUT, exist_ok=True)

workflow = {
  "4": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}},
  "5": {"class_type": "EmptyLatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
  "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["4", 1]}},
  "7": {"class_type": "CLIPTextEncode", "inputs": {"text": negative, "clip": ["4", 1]}},
  "3": {"class_type": "KSampler", "inputs": {"seed": seed, "steps": steps, "cfg": cfg,
        "sampler_name": "euler", "scheduler": "normal", "denoise": 1.0,
        "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}},
  "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["4", 2]}},
  "9": {"class_type": "SaveImage", "inputs": {"filename_prefix": "sdxl", "images": ["8", 0]}},
}

req = urllib.request.Request(f"{SERVER}/prompt",
    data=json.dumps({"prompt": workflow}).encode(),
    headers={"Content-Type": "application/json"})
pid = json.loads(urllib.request.urlopen(req).read())["prompt_id"]
print(f"submitted: {pid}")

t0 = time.time()
LAST = None
while time.time() - t0 < 300:
    try:
        h = json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except Exception:
        h = {}
    if pid in h:
        elapsed = time.time() - t0
        for node_id, out in h[pid]["outputs"].items():
            for item in out.get("images", []):
                q = urllib.parse.urlencode({"filename": item["filename"],
                    "subfolder": item.get("subfolder", ""), "type": "output"})
                data = urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                dest = os.path.join(OUT, item["filename"])
                open(dest, "wb").write(data)
                LAST = dest
                print(f"DONE in {elapsed:.1f}s -> {dest} ({len(data)//1024} KB)")
        break
    time.sleep(1)
else:
    print("TIMEOUT after 300s")


In [ ]:
#@title 4 · Display result { display-mode: "form" }
from IPython.display import Image, display
if LAST:
    display(Image(filename=LAST))
else:
    print("no image yet - run cell 3 first")
